# Concrete Evidence AI — Train YOLOv8-seg on SORD

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Part A pre-trains on SORD (synthetic rebar, classes `intersection` / `spacing`). Part B fine-tunes on our own Roboflow labels (`bar · tie · hook 90° · hook 135°`) once they exist.

In [ ]:
# 1. Check GPU + install
!nvidia-smi -L
!pip install -q ultralytics huggingface_hub pycocotools

In [ ]:
# 2. Save results to Google Drive (so best.pt survives the session)
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/ConcreteEvidence/runs'
import os; os.makedirs(SAVE, exist_ok=True)

In [ ]:
# 3. Get the repo code
%cd /content
!rm -rf AAD_Research_Studio_03 && git clone -q https://github.com/iaac-maai/AAD_Research_Studio_03
%cd AAD_Research_Studio_03

In [ ]:
# 4. Download SORD: N train shards (~480 MB, ~100 images each) + 1 validation shard
N_TRAIN = 3
from huggingface_hub import hf_hub_download
import tarfile
files = [f'SORD-train-{i:05d}.tar' for i in range(N_TRAIN)] + ['SORD-valid-00000.tar']
for f in files:
    if os.path.isdir(f'data/sord/{f[:-4]}'):
        print('have', f); continue
    print('downloading', f)
    p = hf_hub_download('bugwei/synthetic_on-site_rebar_data', f, repo_type='dataset', local_dir='data/sord/_tars')
    with tarfile.open(p) as t: t.extractall('data/sord')
    os.remove(p)
!ls data/sord; ls data/sord/SORD-train-00000 | head

In [ ]:
# 5. Convert COCO JSON -> YOLO-seg labels
import glob
train = ' '.join(sorted(glob.glob('data/sord/SORD-train-*')))
!python training/coco_to_yolo_seg.py --train {train} --val data/sord/SORD-valid-00000 --out data/sord_yolo
!cat data/sord_yolo/data.yaml

In [ ]:
# 6. Look at one labelled image before training
from ultralytics.data.utils import check_det_dataset
import random, cv2, numpy as np, matplotlib.pyplot as plt
img = random.choice(glob.glob('data/sord_yolo/images/train/*'))
lbl = img.replace('/images/','/labels/').rsplit('.',1)[0]+'.txt'
im = cv2.cvtColor(cv2.imread(img), cv2.COLOR_BGR2RGB); h,w = im.shape[:2]
for line in open(lbl).read().split('\n'):
    if not line: continue
    v = list(map(float,line.split())); c=int(v[0]); pts=(np.array(v[1:]).reshape(-1,2)*[w,h]).astype(np.int32)
    cv2.polylines(im,[pts],True,(255,0,0) if c==0 else (0,200,0),4)
plt.figure(figsize=(12,9)); plt.imshow(im); plt.axis('off'); plt.title(os.path.basename(img))

In [ ]:
# 7. TRAIN (yolov8n-seg = smallest/fastest; ~20-40 min on a T4 for 50 epochs)
from ultralytics import YOLO
model = YOLO('yolov8n-seg.pt')
model.train(data='data/sord_yolo/data.yaml', epochs=50, imgsz=1024, batch=8,
            patience=15, project=SAVE, name='sord_pretrain', exist_ok=True)

In [ ]:
# 8. Results + test on validation images
from IPython.display import Image as Show, display
run = f'{SAVE}/sord_pretrain'
display(Show(f'{run}/results.png', width=900))
best = YOLO(f'{run}/weights/best.pt')
for r in best.predict(sorted(glob.glob('data/sord_yolo/images/val/*'))[:3], imgsz=1024, conf=0.25):
    plt.figure(figsize=(12,9)); plt.imshow(r.plot()[:,:,::-1]); plt.axis('off')
print('best.pt saved at', f'{run}/weights/best.pt')

## Part B — fine-tune on our own labels (after labelling in Roboflow)
In Roboflow: Generate version → Export → **YOLOv8** format → *Show download code*. Paste your workspace/project/version below. Never commit the API key.

In [ ]:
# 9. Fine-tune from the SORD weights on our Roboflow dataset
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key='PASTE_KEY')
# ds = rf.workspace('akshat-dayal-students-iaac-net').project('YOUR_PROJECT').version(1).download('yolov8', location='data/ours')
# model = YOLO(f'{SAVE}/sord_pretrain/weights/best.pt')
# model.train(data='data/ours/data.yaml', epochs=100, imgsz=1024, batch=8, project=SAVE, name='ours_finetune')